# autoresearch-medimage - Colab autoresearch session

This notebook runs a real multi-experiment ChestX-ray14 search session in Colab.

What it does:
- uses a named `--session`
- runs a small seed set of fixed-budget recipes
- keeps a frontier of the best kept runs across multiple backbones
- generates mutation waves around that frontier instead of only mutating one winner once
- auto-marks each run as `keep`, `candidate`, or `discard`
- syncs `results.tsv`, run logs, and checkpoints back to Drive
- refreshes `progress.png`, runs a held-out test evaluation for the best run, and backs up the core artifacts at the end

What it is not:
- not a code-mutating overnight agent loop
- not a single-baseline demo
- not a paper or regulatory generation workflow

For a single ChestX-ray14 baseline, use `notebooks/colab_chexnet.ipynb`.
For the fastest smoke test and BYOD flow, use `notebooks/colab_quickstart.ipynb`.


## 1. Setup

In [ ]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

In [ ]:
# Download the public repo snapshot into /content
!rm -rf /content/autoresearch-medimage /content/autoresearch-medimage-main /content/repo.zip
!wget -q https://github.com/mlungrengmail/autoresearch-medimage/archive/refs/heads/main.zip -O /content/repo.zip
!unzip -q /content/repo.zip -d /content
!mv /content/autoresearch-medimage-main /content/autoresearch-medimage
%cd /content/autoresearch-medimage

In [ ]:
# Install dependencies
!uv sync

## 2. Mount Drive and define the session

This notebook expects that you already have a ChestX-ray14 copy in Drive, for example under `MyDrive/chestxray14`.

The default path stages the dataset onto local `/content` SSD once, then reuses it for the whole session. That first copy takes time, but it makes repeated experiments much faster than training through the Drive mount.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
from datetime import datetime

# Update these before running the next cell.
CHESTXRAY14_SOURCE = '/content/drive/MyDrive/chestxray14'
SESSION = f"chestxray14_colab_search_{datetime.now():%Y%m%d_%H%M%S}"
RUNS_ROOT = '/content/drive/MyDrive/autoresearch_runs'
USE_LOCAL_SSD = True
MIN_IMPROVEMENT = 0.10

print('SESSION =', SESSION)

In [ ]:
import os
import shutil
import tarfile
import zipfile
from pathlib import Path

source = Path(CHESTXRAY14_SOURCE)
repo_data = Path('/content/autoresearch-medimage/data/chestxray14')
cache_dir = Path('/content/chestxray14_cache')

if not source.exists():
    raise FileNotFoundError(source)

if repo_data.is_symlink() or repo_data.exists():
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    else:
        repo_data.unlink()

target = source
if USE_LOCAL_SSD or source.is_file():
    if cache_dir.exists():
        print('Reusing existing local cache:', cache_dir)
    else:
        cache_dir.mkdir(parents=True, exist_ok=True)
        if source.is_dir():
            print('Copying ChestX-ray14 from Drive to local SSD. This can take a while on the first run...')
            shutil.copytree(source, cache_dir, dirs_exist_ok=True)
        elif source.suffix.lower() == '.zip':
            with zipfile.ZipFile(source) as archive:
                archive.extractall(cache_dir)
        elif source.name.endswith(('.tar.gz', '.tgz', '.tar')):
            with tarfile.open(source) as archive:
                archive.extractall(cache_dir)
        else:
            raise ValueError('Use a folder, .zip, .tar.gz, .tgz, or .tar archive.')
    target = cache_dir

os.symlink(target, repo_data, target_is_directory=True)
print('linked:', repo_data)
print('points to:', os.readlink(repo_data))

## 3. Prepare dataset metadata for this runtime

Always use `--refresh-splits` here. It rebuilds `splits.csv` with Colab/Linux paths and avoids stale Windows paths from previous local runs.

In [ ]:
!uv run prepare.py --dataset chestxray14 --refresh-splits

## 4. Search helpers

The helpers below seed the session, keep a frontier of the strongest kept runs, generate new mutation waves around that frontier, mark each result, and sync the evolving search state back to Drive.


In [ ]:
import math
import shlex
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from tools.autoresearch_loop import (
    load_checkpoint_config,
    propose_mutation_specs,
    recipe_signature,
    recipe_spec_from_run,
    slugify,
    write_recipe_file,
)

REPO_DIR = Path('/content/autoresearch-medimage')
RUNS_DIR = Path(RUNS_ROOT) / SESSION
RUNLOG_DIR = REPO_DIR / 'runlogs'
RECIPE_DIR = REPO_DIR / 'recipes'
RUNLOG_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def session_frame():
    results_path = REPO_DIR / 'results.tsv'
    if not results_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(results_path, sep='\t')
    view = df[df['dataset'].astype(str) == 'chestxray14'].copy()
    if 'session' in view.columns:
        view = view[view['session'].fillna('').astype(str) == SESSION].copy()
    if 'metric' in view.columns:
        view['metric'] = pd.to_numeric(view['metric'], errors='coerce')
    return view

def best_keep_metric():
    view = session_frame()
    if view.empty or 'status' not in view.columns:
        return None
    keep = view[view['status'].astype(str) == 'keep']
    if keep.empty:
        return None
    return float(keep['metric'].max())

def best_keep_row():
    view = session_frame()
    if view.empty:
        return None
    if 'status' not in view.columns:
        return view.sort_values('metric', ascending=False).iloc[0]
    keep = view[view['status'].astype(str) == 'keep']
    if keep.empty:
        return None
    return keep.sort_values('metric', ascending=False).iloc[0]

def best_any_row():
    view = session_frame()
    if view.empty:
        return None
    return view.sort_values('metric', ascending=False).iloc[0]

def frontier_parent_rows(max_parents=3):
    view = session_frame()
    if view.empty:
        return []
    ranked = view.copy()
    status_order = {'keep': 0, 'candidate': 1, 'discard': 2, 'crash': 3}
    if 'status' in ranked.columns:
        ranked['__status_rank'] = ranked['status'].astype(str).map(lambda s: status_order.get(s, 9))
    else:
        ranked['__status_rank'] = 1
    ranked = ranked.sort_values(['__status_rank', 'metric'], ascending=[True, False])

    chosen = []
    used_backbones = set()
    used_recipes = set()

    for _, row in ranked.iterrows():
        status = str(row.get('status', 'candidate'))
        if status not in {'keep', 'candidate'}:
            continue
        backbone = str(row.get('backbone', '')).strip()
        recipe = str(row.get('recipe', '')).strip()
        if backbone and backbone not in used_backbones:
            chosen.append(row)
            used_backbones.add(backbone)
            if recipe:
                used_recipes.add(recipe)
        if len(chosen) >= max_parents:
            return chosen

    for _, row in ranked.iterrows():
        status = str(row.get('status', 'candidate'))
        if status not in {'keep', 'candidate'}:
            continue
        recipe = str(row.get('recipe', '')).strip()
        if recipe and recipe in used_recipes:
            continue
        chosen.append(row)
        if recipe:
            used_recipes.add(recipe)
        if len(chosen) >= max_parents:
            break
    return chosen

def sync_session_state():
    shutil.copy2(REPO_DIR / 'results.tsv', RUNS_DIR / 'results.tsv')
    ckpt_src = REPO_DIR / 'checkpoints'
    if ckpt_src.exists():
        ckpt_dst = RUNS_DIR / 'checkpoints'
        if ckpt_dst.exists():
            shutil.rmtree(ckpt_dst)
        shutil.copytree(ckpt_src, ckpt_dst)
    log_src = REPO_DIR / 'runlogs'
    if log_src.exists():
        log_dst = RUNS_DIR / 'runlogs'
        if log_dst.exists():
            shutil.rmtree(log_dst)
        shutil.copytree(log_src, log_dst)

def run_logged(command, log_path):
    print('+', ' '.join(shlex.quote(str(x)) for x in command))
    with open(log_path, 'w', encoding='utf-8') as log_handle:
        proc = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end='')
            log_handle.write(line)
        returncode = proc.wait()
    if returncode != 0:
        raise RuntimeError(f'Command failed with exit code {returncode}: {command}')

def mark_latest(status, note=''):
    command = [
        'uv', 'run', 'python', 'tools/mark_result.py',
        '--dataset', 'chestxray14',
        '--session', SESSION,
        '--status', status,
    ]
    if note:
        command.extend(['--append-note', note])
    subprocess.run(command, cwd=REPO_DIR, check=True)

def run_search_entry(label, args):
    before = best_keep_metric()
    log_path = RUNLOG_DIR / f"{int(time.time())}_{label}.log"
    command = ['uv', 'run', 'train.py', '--dataset', 'chestxray14', '--session', SESSION, '--save-checkpoint'] + list(args)
    run_logged(command, log_path)
    latest = session_frame().iloc[-1]
    metric = float(latest['metric'])
    if before is None:
        status = 'keep'
        note = 'session baseline'
    elif metric >= before + MIN_IMPROVEMENT:
        status = 'keep'
        note = f'improved by {metric - before:.3f}'
    elif metric >= before:
        status = 'candidate'
        note = f'within {MIN_IMPROVEMENT:.2f} of best keep'
    else:
        status = 'discard'
        note = f'below best keep by {before - metric:.3f}'
    mark_latest(status, note)
    sync_session_state()
    latest = session_frame().iloc[-1]
    print('\nDecision:', status, '| metric =', metric)
    return latest

def existing_recipe_signatures():
    signatures = set()
    view = session_frame()
    if view.empty:
        return signatures
    for _, row in view.iterrows():
        ckpt_config = load_checkpoint_config(row.get('checkpoint_path', ''))
        spec = recipe_spec_from_run(row, checkpoint_config=ckpt_config, dataset_name='chestxray14')
        signatures.add(recipe_signature(spec))
    return signatures

def unseen_seed_entries():
    view = session_frame()
    seen = set(view['recipe'].astype(str)) if not view.empty and 'recipe' in view.columns else set()
    return [entry for entry in SEED_PLAN if entry['args'][1] not in seen]

def build_mutation_entries_for_parent(parent_row, wave_index=0, limit=2):
    ckpt_config = load_checkpoint_config(parent_row.get('checkpoint_path', ''))
    base_spec = recipe_spec_from_run(parent_row, checkpoint_config=ckpt_config, dataset_name='chestxray14')
    taken = existing_recipe_signatures()
    entries = []
    base_slug = slugify(parent_row.get('backbone', 'candidate'))
    parent_recipe = slugify(parent_row.get('recipe', base_slug))
    for label, spec in propose_mutation_specs(base_spec, wave_index=wave_index):
        signature = recipe_signature(spec)
        if signature in taken:
            continue
        recipe_name = f"colab_{slugify(SESSION)}_{base_slug}_{parent_recipe}_w{wave_index}_{slugify(label)}"
        suffix = 1
        while (RECIPE_DIR / f"{recipe_name}.yaml").exists():
            recipe_name = f"colab_{slugify(SESSION)}_{base_slug}_{parent_recipe}_w{wave_index}_{slugify(label)}_{suffix}"
            suffix += 1
        write_recipe_file(RECIPE_DIR, recipe_name, spec)
        entries.append({
            'label': f"{parent_row.get('backbone', 'model')}::{label}",
            'args': ['--recipe', recipe_name],
            'parent_backbone': str(parent_row.get('backbone', '')),
            'parent_metric': float(parent_row.get('metric', 0.0)),
        })
        taken.add(signature)
        if len(entries) >= limit:
            break
    return entries

def build_next_mutation_queue(start_wave=0, limit=6, max_waves=12, max_parents=3):
    wave = start_wave
    for _ in range(max_waves):
        parents = frontier_parent_rows(max_parents=max_parents)
        if not parents:
            return [], wave, 0
        per_parent_limit = max(1, math.ceil(limit / max(1, len(parents))))
        proposal_lists = [
            build_mutation_entries_for_parent(parent_row, wave_index=wave, limit=per_parent_limit)
            for parent_row in parents
        ]
        queue = []
        while len(queue) < limit and any(proposal_lists):
            for proposals in proposal_lists:
                if proposals and len(queue) < limit:
                    queue.append(proposals.pop(0))
        wave += 1
        if queue:
            return queue, wave, len(parents)
    return [], wave, 0

def run_autoresearch_loop(target_total_runs=16, max_stalled_waves=2, mutation_batch_size=6, max_frontier_parents=3):
    results = []
    queue = unseen_seed_entries()
    queue_kind = 'seed' if queue else None
    wave_index = max(0, len(session_frame()) - len(SEED_PLAN)) // max(1, mutation_batch_size)
    stalled_waves = 0
    while len(session_frame()) < target_total_runs:
        if not queue:
            queue, wave_index, parent_count = build_next_mutation_queue(
                start_wave=wave_index,
                limit=mutation_batch_size,
                max_parents=max_frontier_parents,
            )
            queue_kind = 'mutation'
            if not queue:
                print('No unseen mutation waves left for the current frontier.')
                break
            print(f'\nStarting mutation wave {wave_index} across {parent_count} frontier parent(s).')
        before_wave_best = best_keep_metric() if queue_kind == 'mutation' else None
        runs_this_queue = 0
        while queue and len(session_frame()) < target_total_runs:
            entry = queue.pop(0)
            print('\n' + '=' * 80)
            print('RUNNING', entry['label'])
            print('=' * 80)
            results.append(run_search_entry(entry['label'], entry['args']))
            runs_this_queue += 1
        if queue_kind == 'mutation' and runs_this_queue > 0:
            after_wave_best = best_keep_metric()
            if before_wave_best is None or (after_wave_best is not None and after_wave_best > before_wave_best + 1e-9):
                stalled_waves = 0
            else:
                stalled_waves += 1
                print(f'Stalled mutation waves: {stalled_waves}/{max_stalled_waves}')
                if stalled_waves >= max_stalled_waves:
                    print(f'Stopping after {stalled_waves} consecutive non-improving mutation waves.')
                    break
        queue_kind = None
    view = session_frame().sort_values('metric', ascending=False)
    display(view[['timestamp', 'recipe', 'backbone', 'metric', 'status']])
    return results

def restore_session_from_drive(existing_session):
    source_dir = Path(RUNS_ROOT) / existing_session
    if not source_dir.exists():
        raise FileNotFoundError(source_dir)
    if (source_dir / 'results.tsv').exists():
        shutil.copy2(source_dir / 'results.tsv', REPO_DIR / 'results.tsv')
    if (source_dir / 'checkpoints').exists():
        dst = REPO_DIR / 'checkpoints'
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(source_dir / 'checkpoints', dst)
    if (source_dir / 'runlogs').exists():
        dst = REPO_DIR / 'runlogs'
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(source_dir / 'runlogs', dst)
    print('restored session from', source_dir)


## 5. Optional: restore a previous session after a runtime reset

In [ ]:
# Set this only if you want to resume a previous Drive-backed session.
RESTORE_SESSION = ''
if RESTORE_SESSION:
    restore_session_from_drive(RESTORE_SESSION)

## 6. Seed recipes and continuing frontier search loop

The session starts with a small seed set, then keeps generating mutation waves around the strongest kept frontier instead of tunneling on only one winner.

Keep the 300-second contract fixed. Increase `TARGET_TOTAL_RUNS` if you want the search to keep going longer in the same session.


In [ ]:
SEED_PLAN = [
    {'label': 'baseline_densenet121', 'args': ['--recipe', 'chexnet']},
    {'label': 'advanced_densenet121', 'args': ['--recipe', 'chexnet_advanced']},
    {'label': 'resnet50', 'args': ['--recipe', 'chexnet_resnet50']},
    {'label': 'efficientnet_b0', 'args': ['--recipe', 'chexnet_efficientnet_b0']},
    {'label': 'convnext_tiny', 'args': ['--recipe', 'chexnet_convnext_tiny']},
    # Optional heavier run if you want to try a medical foundation model:
    # {'label': 'biomedclip', 'args': ['--recipe', 'chexnet_biomedclip']},
]

TARGET_TOTAL_RUNS = 16
MAX_STALLED_WAVES = 2
MAX_FRONTIER_PARENTS = 3
MUTATIONS_PER_WAVE = 6

print('Seed runs:', len(SEED_PLAN))
print('Target total runs:', TARGET_TOTAL_RUNS)
print('Frontier parents per wave:', MAX_FRONTIER_PARENTS)
print('Mutation proposals per wave:', MUTATIONS_PER_WAVE)
print('Max stalled waves:', MAX_STALLED_WAVES)


In [ ]:
results = run_autoresearch_loop(
    target_total_runs=TARGET_TOTAL_RUNS,
    max_stalled_waves=MAX_STALLED_WAVES,
    mutation_batch_size=MUTATIONS_PER_WAVE,
    max_frontier_parents=MAX_FRONTIER_PARENTS,
)


In [ ]:
# Review the session leaderboard at any point
view = session_frame().sort_values('metric', ascending=False)
display(view[['timestamp', 'recipe', 'backbone', 'metric', 'status', 'checkpoint_path']])

## 7. Evaluate and back up the best run

This ending stays focused on the core autoresearch outputs:
- refresh `progress.png`
- run a held-out test evaluation for the current winning checkpoint
- back up `results.tsv`, `progress.png`, the held-out eval payload, checkpoints, and run logs to Drive

It intentionally skips paper and regulatory generation.


In [ ]:
# Refresh the progress plot and evaluate the current winner on the held-out test split
# The held-out evaluation step is usually the slowest part of this notebook.
view = session_frame().sort_values('metric', ascending=False)
keep = view[view['status'].astype(str) == 'keep'].copy()
winner = keep.iloc[0] if not keep.empty else view.iloc[0]
WINNER_CKPT = str(winner['checkpoint_path'])

print('winner metric:', winner['metric'])
print('winner recipe:', winner['recipe'])
print('winner checkpoint:', WINNER_CKPT)

!uv run python tools/progress_plot.py --results /content/autoresearch-medimage/results.tsv --dataset chestxray14 --session "{SESSION}" --output /content/autoresearch-medimage/progress.png
!uv run python tools/evaluate_checkpoint.py --checkpoint "{WINNER_CKPT}" --split test --output /content/autoresearch-medimage/paper/generated/chestxray14_final_eval.pt


In [ ]:
# Back up the core autoresearch artifacts to Drive
final_dir = Path(RUNS_ROOT) / SESSION / 'final_artifacts'
final_dir.mkdir(parents=True, exist_ok=True)

for candidate in [
    REPO_DIR / 'results.tsv',
    REPO_DIR / 'progress.png',
    REPO_DIR / 'paper' / 'generated' / 'chestxray14_final_eval.pt',
]:
    if candidate.exists():
        shutil.copy2(candidate, final_dir / candidate.name)

for folder_name in ['checkpoints', 'runlogs']:
    src = REPO_DIR / folder_name
    if src.exists():
        dst = final_dir / folder_name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)

print('Backed up core autoresearch artifacts to', final_dir)
